<a href="https://colab.research.google.com/github/yiyu-chen-labs/llm-from-scratch/blob/main/day27_qlora_memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U torchao bitsandbytes transformers peft accelerate

import torch, transformers, peft, bitsandbytes
print("torch         ", torch.__version__)
print("transformers  ", transformers.__version__)
print("peft          ", peft.__version__)
print("bitsandbytes  ", bitsandbytes.__version__)
print("GPU           ", torch.cuda.get_device_name(0))
print("VRAM          ", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 28.4 MB/s eta 0:00:00


torch          2.11.0+cu128
transformers   5.15.0
peft           0.20.0
bitsandbytes   0.50.1
GPU            Tesla T4
VRAM           15.6 GB


In [ ]:
import gc, time, torch

MODEL   = "Qwen/Qwen2.5-0.5B-Instruct"
SEQ_LEN = 256
BATCH   = 1
STEPS   = 3

def clear():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

def peak_gb():
    return torch.cuda.max_memory_allocated() / 1e9

In [ ]:
def run_config(name, load_fn):
    clear()

    model, tok = load_fn()
    load_peak = peak_gb()

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    opt = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=1e-4
    )

    ids = torch.randint(0, 1000, (BATCH, SEQ_LEN), device="cuda")
    batch = {"input_ids": ids, "labels": ids.clone()}

    model(**batch).loss.backward()
    opt.step()
    opt.zero_grad(set_to_none=True)
    torch.cuda.synchronize()

    t0 = time.time()
    for _ in range(STEPS):
        model(**batch).loss.backward()
        opt.step()
        opt.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    ms = (time.time() - t0) / STEPS * 1000

    r = dict(name=name, load_gb=load_peak, peak_gb=peak_gb(),
             trainable=trainable, ms_per_step=ms)

    del model, opt, batch, ids
    clear()
    return r

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

tok = AutoTokenizer.from_pretrained(MODEL)

LORA = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

def load_full():
    m = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).cuda()
    return m, tok

def load_lora():
    m = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16).cuda()
    return get_peft_model(m, LORA), tok

def load_qlora():
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    m = AutoModelForCausalLM.from_pretrained(
        MODEL, quantization_config=bnb, device_map={"": 0}
    )
    m = prepare_model_for_kbit_training(m, use_gradient_checkpointing=False)
    return get_peft_model(m, LORA), tok

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [ ]:
results = []
for name, fn in [("Full FT (fp32)", load_full),
                 ("LoRA (fp16)",    load_lora),
                 ("QLoRA (nf4)",    load_qlora)]:
    r = run_config(name, fn)
    results.append(r)
    print(f"{r['name']:<16} load {r['load_gb']:5.2f} GB   "
          f"peak {r['peak_gb']:5.2f} GB   "
          f"trainable {r['trainable']:>12,}   "
          f"{r['ms_per_step']:6.0f} ms/step")

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Full FT (fp32)   load  1.98 GB   peak  9.93 GB   trainable  494,032,768      410 ms/step


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

LoRA (fp16)      load  1.02 GB   peak  2.00 GB   trainable    1,081,344      143 ms/step


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

QLoRA (nf4)      load  1.02 GB   peak  1.90 GB   trainable    1,081,344      218 ms/step


In [ ]:
import pandas as pd

df = pd.DataFrame(results)
base_mem = df.loc[0, "peak_gb"]
base_ms  = df.loc[0, "ms_per_step"]

df["記憶體省下"] = (1 - df["peak_gb"] / base_mem).map(lambda v: f"{v*100:.0f}%")
df["相對速度"]   = (df["ms_per_step"] / base_ms).map(lambda v: f"{v:.2f}x")

out = df[["name", "load_gb", "peak_gb", "記憶體省下", "trainable", "ms_per_step", "相對速度"]]
out.columns = ["組別", "載入後 (GB)", "訓練峰值 (GB)", "記憶體省下", "可訓練參數", "ms/step", "相對速度"]
print(out.to_markdown(index=False, floatfmt=".2f"))

| 組別           |   載入後 (GB) |   訓練峰值 (GB) | 記憶體省下   |   可訓練參數 |   ms/step | 相對速度   |
|:---------------|--------------:|----------------:|:-------------|-------------:|----------:|:-----------|
| Full FT (fp32) |          1.98 |            9.93 | 0%           |    494032768 |    409.88 | 1.00x      |
| LoRA (fp16)    |          1.02 |            2.00 | 80%          |      1081344 |    143.10 | 0.35x      |
| QLoRA (nf4)    |          1.02 |            1.90 | 81%          |      1081344 |    217.97 | 0.53x      |


In [ ]:
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
m = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map={"": 0})
w = m.model.layers[0].self_attn.q_proj.weight

print("權重類別      ", type(w).__name__)
print("dtype         ", w.dtype)
print("存起來的 shape ", tuple(w.shape))
print("實際佔用 bytes ", w.numel() * w.element_size())

qs = w.quant_state
print("量化型別      ", qs.quant_type)
print("blocksize     ", qs.blocksize)
print("absmax 數量   ", qs.absmax.numel())
print("absmax dtype  ", qs.absmax.dtype, "  <- double quant 後不是 fp32")

orig = qs.shape
print()
print(f"原始矩陣      {tuple(orig)}  = {orig[0]*orig[1]:,} 個權重")
print(f"fp16 需要     {orig[0]*orig[1]*2/1e6:.2f} MB")
print(f"nf4  實際用   {w.numel()*w.element_size()/1e6:.2f} MB")

del m, w
clear()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

權重類別       Params4bit
dtype          torch.uint8
存起來的 shape  (401408, 1)
實際佔用 bytes  401408
量化型別       nf4
blocksize      64
absmax 數量    12544
absmax dtype   torch.uint8   <- double quant 後不是 fp32

原始矩陣      (896, 896)  = 802,816 個權重
fp16 需要     1.61 MB
nf4  實際用   0.40 MB


In [ ]:
def project(n_params, lora_frac=0.005):
    adapter = n_params * lora_frac * 16 / 1e9
    return {
        "Full FT": n_params * 16 / 1e9,
        "LoRA":    n_params * 2  / 1e9 + adapter,
        "QLoRA":   n_params * 0.5625 / 1e9 + adapter,
    }

print(f"{'模型':<10}{'Full FT':>12}{'LoRA':>12}{'QLoRA':>12}   {'單張 24GB 卡':>16}")
for label, n in [("0.5B", 0.5e9), ("7B", 7e9), ("13B", 13e9), ("70B", 70e9)]:
    p = project(n)
    fits = [k for k, v in p.items() if v < 22]
    print(f"{label:<10}{p['Full FT']:>10.1f}GB{p['LoRA']:>10.1f}GB{p['QLoRA']:>10.1f}GB"
          f"   {', '.join(fits) if fits else '都塞不下':>16}")

模型             Full FT        LoRA       QLoRA          單張 24GB 卡
0.5B             8.0GB       1.0GB       0.3GB   Full FT, LoRA, QLoRA
7B             112.0GB      14.6GB       4.5GB        LoRA, QLoRA
13B            208.0GB      27.0GB       8.4GB              QLoRA
70B           1120.0GB     145.6GB      45.0GB               都塞不下
